# Colab Subtitle Gateway

这个 notebook 是**唯一的部署入口与配置面**。直接编辑下一格 `CONFIG`，再从本机执行：

```bash
colab new --gpu T4 -s colab-sg                  # 首次创建会话；已有会话时跳过
colab exec -s colab-sg -f colab-sg.ipynb --timeout 21600
```

也可以在 Colab 网页里打开 notebook 后运行全部单元格。没有交互式向导；`mode` 控制部署、状态、停止或卸载。部署进程由 Python `subprocess` 独立启动，不依赖 notebook kernel 持续存活。

> **密钥约定**：按本项目配置，密钥直接放在下方 notebook，并会进入 Git 提交历史。不要把仓库公开；若曾公开，立即轮换密钥。执行输出不打印密钥，但网关启动参数仍可被同一 VM 上的进程列表看到。
>
> Cloudflare tunnel 由 Python 下载并管理 Cloudflare 官方 `cloudflared` 程序。Quick、connector token 和 cert.pem named tunnel 均可配置；Python 库本身并不替代 Cloudflare 的 tunnel connector。


In [ ]:
# 唯一配置入口。编辑这里，然后按需重跑整个 notebook。
CONFIG = {
    "mode": "deploy",                 # deploy | status | stop | uninstall
    "session": "colab-sg",            # 本机 colab CLI 与服务模板使用
    "gpu": "T4",                      # T4 / L4 / G4 / A100 / H100
    "port": 8000,
    "device": "cuda",                 # cuda | auto | cpu
    "root": "/content/csg",           # Colab VM 内的工作目录
    "upstream_repo": "https://github.com/canxin121/subtitle-gateway.git",
    "upstream_ref": "main",
    "models": ["fun-asr-mlt-nano"],
    "preload": ["fun-asr-mlt-nano"],
    "max_loaded_models": 2,            # 0 = 不限；None = gateway 默认值
    "protocols": ["openai", "ferrum", "deepl", "libretranslate"],
    "translate_free": "google,edge",  # none 可真正关闭免费翻译源
    "translate_upstream": "",         # DeepL-compatible 上游 URL
    "translate_upstream_key": "",     # 上游 key
    "translate_api_key": "",          # /v1/translate 客户端鉴权 key
    "libretranslate_upstream": "",
    "libretranslate_upstream_key": "",
    "libretranslate_api_key": "",    # /translate 客户端鉴权 key
    "auth_secret": "",                # ferrum: 客户端 x-auth-token = hex(sha256(secret))
    "encryption_key": "",             # ferrum AES passphrase
    "tunnel": "quick",                # quick | token | named | off
    "tunnel_name": "",                # named tunnel 名称 (需要 VM 内 cert.pem)
    "tunnel_hostname": "",            # 固定域名，不含 https://；token/named 使用
    "tunnel_token": "",               # connector token；会写入 VM 上 600 权限文件
    "pip_index": "",                  # 可选 pip index URL
    "proxy": "",                      # 可选 HTTP(S) 代理 URL
    "force_deps": False,
    "force_download": False,
    "run_demo": False,
    "confirm_uninstall": False,         # mode=uninstall 时还需显式改为 True
}


In [ ]:
import copy
import ctypes.util
import hashlib
import json
import os
import platform
import re
import shlex
import shutil
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path


def _split_values(value):
    return [part for part in re.split(r"[,\s]+", value.strip()) if part]


def _as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"1", "true", "yes", "on"}


SETTINGS = dict(CONFIG)
_ENV_FIELDS = {
    "mode": ("CSG_MODE", str),
    "session": ("CSG_SESSION", str),
    "gpu": ("CSG_GPU", str),
    "port": ("CSG_PORT", int),
    "device": ("CSG_DEVICE", str),
    "root": ("CSG_ROOT", str),
    "models": ("CSG_MODELS", _split_values),
    "preload": ("CSG_PRELOAD", _split_values),
    "max_loaded_models": ("CSG_MAX_LOADED_MODELS", lambda v: None if not v.strip() else int(v)),
    "protocols": ("CSG_PROTOCOLS", _split_values),
    "translate_free": ("CSG_TRANSLATE_FREE", str),
    "translate_upstream": ("CSG_TRANSLATE_UPSTREAM", str),
    "translate_upstream_key": ("CSG_TRANSLATE_UPSTREAM_KEY", str),
    "translate_api_key": ("CSG_TRANSLATE_API_KEY", str),
    "libretranslate_upstream": ("CSG_LIBRETRANSLATE_UPSTREAM", str),
    "libretranslate_upstream_key": ("CSG_LIBRETRANSLATE_UPSTREAM_KEY", str),
    "libretranslate_api_key": ("CSG_LIBRETRANSLATE_API_KEY", str),
    "auth_secret": ("CSG_AUTH_SECRET", str),
    "encryption_key": ("CSG_ENCRYPTION_KEY", str),
    "tunnel": ("CSG_TUNNEL_MODE", str),
    "tunnel_name": ("CSG_TUNNEL_NAME", str),
    "tunnel_hostname": ("CSG_TUNNEL_HOSTNAME", str),
    "tunnel_token": ("CSG_TUNNEL_TOKEN", str),
    "pip_index": ("CSG_PIP_INDEX", str),
    "proxy": ("CSG_PROXY", str),
    "force_deps": ("CSG_FORCE_DEPS", _as_bool),
    "force_download": ("CSG_FORCE_DOWNLOAD", _as_bool),
    "run_demo": ("CSG_RUN_DEMO", _as_bool),
}
for _field, (_env_name, _convert) in _ENV_FIELDS.items():
    if _env_name in os.environ:
        SETTINGS[_field] = _convert(os.environ[_env_name])

MODE = str(SETTINGS["mode"]).strip().lower()
if MODE not in {"deploy", "status", "stop", "uninstall"}:
    raise ValueError("CONFIG['mode'] must be deploy, status, stop, or uninstall")

ROOT = Path(SETTINGS["root"]).expanduser()
if not ROOT.is_absolute():
    ROOT = (Path.cwd() / ROOT).resolve()
else:
    ROOT = ROOT.resolve()
REPO = ROOT / "repo"
RUN = ROOT / "run"
SELECTED_MANIFEST = ROOT / "models.selected.json"
UPSTREAM_MANIFEST = ROOT / "models.upstream.json"
DEPLOY_ENV = ROOT / "deploy.env"
SECRETS_ENV = ROOT / "secrets.env"
TUNNEL_TOKEN_FILE = ROOT / "tunnel.token"
CLOUDFLARED = ROOT / "cloudflared"
PYTHON = REPO / ".venv" / "bin" / "python"
PORT = int(SETTINGS["port"])
PROTOCOLS = set(SETTINGS["protocols"])

if MODE == "deploy":
    if not 1 <= PORT <= 65535:
        raise ValueError("port must be between 1 and 65535")
    if not SETTINGS["models"]:
        raise ValueError("CONFIG['models'] must contain at least one model id")
    unknown_protocols = PROTOCOLS - {"openai", "ferrum", "deepl", "libretranslate"}
    if unknown_protocols:
        raise ValueError(f"Unknown protocol(s): {sorted(unknown_protocols)}")
    if SETTINGS["tunnel"] not in {"quick", "token", "named", "off"}:
        raise ValueError("CONFIG['tunnel'] must be quick, token, named, or off")
    if SETTINGS["device"] not in {"cuda", "auto", "cpu"}:
        raise ValueError("CONFIG['device'] must be cuda, auto, or cpu")
    if SETTINGS["max_loaded_models"] is not None and int(SETTINGS["max_loaded_models"]) < 0:
        raise ValueError("max_loaded_models must be >= 0")
    if SETTINGS["tunnel"] == "token":
        if not str(SETTINGS["tunnel_token"]).strip() or not str(SETTINGS["tunnel_hostname"]).strip():
            raise ValueError("token tunnel requires tunnel_token and tunnel_hostname")
    if SETTINGS["tunnel"] == "named":
        if not str(SETTINGS["tunnel_name"]).strip() or not str(SETTINGS["tunnel_hostname"]).strip():
            raise ValueError("named tunnel requires tunnel_name and tunnel_hostname")
    if "://" in str(SETTINGS["tunnel_hostname"]):
        raise ValueError("tunnel_hostname must be a hostname without an https:// prefix")

_protected_roots = {Path("/").resolve(), Path("/content").resolve(), Path.home().resolve()}
if ROOT in _protected_roots:
    raise ValueError(f"Refusing to use protected root directory: {ROOT}")

RUNTIME_ENV = os.environ.copy()
if SETTINGS.get("proxy"):
    for _name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        RUNTIME_ENV[_name] = SETTINGS["proxy"]
        os.environ[_name] = SETTINGS["proxy"]


def write_private_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    flags = os.O_WRONLY | os.O_CREAT | os.O_TRUNC | getattr(os, "O_NOFOLLOW", 0)
    descriptor = os.open(temp, flags, 0o600)
    with os.fdopen(descriptor, "w", encoding="utf-8") as output:
        output.write(text)
    os.replace(temp, path)
    os.chmod(path, 0o600)


def write_deploy_env():
    fields = {
        "CSG_ROOT": str(ROOT),
        "CSG_PORT": str(PORT),
        "CSG_DEVICE": str(SETTINGS["device"]),
        "CSG_MODELS": " ".join(SETTINGS["models"]),
        "CSG_PRELOAD": " ".join(SETTINGS["preload"]),
        "CSG_PROTOCOLS": " ".join(sorted(PROTOCOLS)),
        "CSG_TRANSLATE_FREE": str(SETTINGS["translate_free"]),
        "CSG_TUNNEL_MODE": str(SETTINGS["tunnel"]),
        "CSG_TUNNEL_NAME": str(SETTINGS["tunnel_name"]),
        "CSG_TUNNEL_HOSTNAME": str(SETTINGS["tunnel_hostname"]),
    }
    text = "# Generated from colab-sg.ipynb CONFIG; do not put secrets here.\n"
    text += "".join(f"{key}={shlex.quote(value)}\n" for key, value in fields.items())
    write_private_text(DEPLOY_ENV, text)


if MODE == "deploy":
    ROOT.mkdir(parents=True, exist_ok=True)
    RUN.mkdir(parents=True, exist_ok=True)
    write_deploy_env()

print(f"mode={MODE} | root={ROOT} | session={SETTINGS['session']} | port={PORT}")
print("configuration loaded (secret values redacted)")


In [ ]:
if MODE == "deploy" and "ferrum" in PROTOCOLS:
    if not ctypes.util.find_library("opus") or not ctypes.util.find_library("opusfile"):
        print("Installing system Opus libraries for the ferrum protocol ...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "libopus0", "libopusfile0"], check=True)
    else:
        print("Opus libraries already available")


In [ ]:
if MODE == "deploy":
    UPSTREAM_URL = str(SETTINGS["upstream_repo"])
    UPSTREAM_REF = str(SETTINGS["upstream_ref"])
    if REPO.exists():
        if not (REPO / ".git").exists():
            raise RuntimeError(f"{REPO} exists but is not an upstream Git checkout; move it or change CONFIG['root']")
        print(f"Updating subtitle-gateway ({UPSTREAM_REF}) ...")
        subprocess.run(
            ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", UPSTREAM_REF],
            env=RUNTIME_ENV,
            check=True,
        )
        subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
    else:
        print(f"Cloning subtitle-gateway ({UPSTREAM_REF}) ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", UPSTREAM_REF, UPSTREAM_URL, str(REPO)],
            env=RUNTIME_ENV,
            check=True,
        )

    # Read the pristine manifest from the Git object, never from the working tree:
    # repo/models.json is overwritten below with the user's selected subset.
    _manifest_bytes = subprocess.check_output(
        ["git", "-C", str(REPO), "show", "HEAD:models.json"],
        stderr=subprocess.STDOUT,
    )
    UPSTREAM_MANIFEST.write_bytes(_manifest_bytes)
    _revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip()
    _upstream = json.loads(_manifest_bytes)
    print(f"upstream revision={_revision} | models={', '.join(_upstream.get('models', {}))}")


In [ ]:
if MODE == "deploy":
    _source_models = UPSTREAM_MANIFEST.read_text(encoding="utf-8")
    _source_manifest = json.loads(_source_models)
    _available = _source_manifest.get("models", {})
    _requested = list(dict.fromkeys(SETTINGS["models"]))
    _missing_ids = [model_id for model_id in _requested if model_id not in _available]
    if _missing_ids:
        print(f"WARNING: upstream no longer contains configured model id(s): {', '.join(_missing_ids)}")
    _selected_ids = [model_id for model_id in _requested if model_id in _available]
    if not _selected_ids:
        raise ValueError(f"None of the configured models exist upstream. Available: {', '.join(_available)}")

    _preload = list(dict.fromkeys(SETTINGS["preload"]))
    _bad_preload = [model_id for model_id in _preload if model_id not in _selected_ids]
    if _bad_preload:
        raise ValueError(f"preload model(s) must also be selected in CONFIG['models']: {_bad_preload}")
    SETTINGS["models"] = _selected_ids
    SETTINGS["preload"] = _preload
    write_deploy_env()

    SELECTED_MANIFEST_DATA = copy.deepcopy(_source_manifest)
    SELECTED_MANIFEST_DATA["models"] = {model_id: _available[model_id] for model_id in _selected_ids}
    SELECTED_MANIFEST_DATA["preload"] = _preload
    _serialized_manifest = json.dumps(SELECTED_MANIFEST_DATA, ensure_ascii=False, indent=2) + "\n"
    write_private_text(SELECTED_MANIFEST, _serialized_manifest)
    (REPO / "models.json").write_text(_serialized_manifest, encoding="utf-8")
    print(f"Selected models: {', '.join(_selected_ids)} | preload: {', '.join(_preload) or '(none)'}")

    _needs_qwen = any(
        model_id.lower().startswith("qwen") or "qwen" in str(_available[model_id].get("model", "")).lower()
        for model_id in _selected_ids
    )


In [ ]:
if MODE == "deploy":
    PYTHON_VERSION = subprocess.check_output([sys.executable, "--version"], text=True).strip()
    print(f"Python runtime: {PYTHON_VERSION}")
    if sys.version_info < (3, 10):
        raise RuntimeError("subtitle-gateway requires Python 3.10 or newer")

    _venv_dir = REPO / ".venv"
    if not _venv_dir.exists():
        print("Creating virtual environment ...")
        subprocess.run([sys.executable, "-m", "venv", str(_venv_dir)], check=True)

    def _pip_prefix():
        return ["--index-url", SETTINGS["pip_index"]] if SETTINGS.get("pip_index") else []

    _import_check = "import funasr, torch" + ("; import qwen_asr" if _needs_qwen else "")
    _deps_ready = subprocess.run(
        [str(PYTHON), "-c", _import_check],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0

    if SETTINGS["force_deps"] or not _deps_ready:
        print("Installing server requirements ...")
        subprocess.run(
            [str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", *_pip_prefix()],
            env=RUNTIME_ENV,
            check=True,
        )
        subprocess.run(
            [str(PYTHON), "-m", "pip", "install", *_pip_prefix(), "-r", str(REPO / "requirements.txt")],
            env=RUNTIME_ENV,
            check=True,
        )
        print("Installing FunASR with compatible transformers/tokenizers pins ...")
        subprocess.run(
            [
                str(PYTHON), "-m", "pip", "install", *_pip_prefix(),
                "funasr", "huggingface_hub", "transformers>=4.49,<5",
                "tokenizers>=0.21", "numpy>=1.26,<3",
            ],
            env=RUNTIME_ENV,
            check=True,
        )
        print("Installing pinned torch and torchaudio ...")
        subprocess.run(
            [str(PYTHON), "-m", "pip", "install", *_pip_prefix(), "torch==2.13.0", "torchaudio==2.11.0"],
            env=RUNTIME_ENV,
            check=True,
        )
        if _needs_qwen:
            # Qwen pins must remain last: qwen-asr requires transformers 4.57.6.
            print("Installing the Qwen3-ASR runtime ...")
            subprocess.run(
                [
                    str(PYTHON), "-m", "pip", "install", *_pip_prefix(),
                    "qwen-asr==0.0.6", "transformers==4.57.6",
                    "tokenizers==0.22.2", "huggingface_hub==0.36.2",
                ],
                env=RUNTIME_ENV,
                check=True,
            )
    else:
        print("Python dependencies already import successfully; skipping installs")

    _cuda_result = subprocess.run(
        [str(PYTHON), "-c", "import torch; print(torch.cuda.is_available())"],
        text=True,
        capture_output=True,
        check=True,
    )
    print(f"torch.cuda.is_available() = {_cuda_result.stdout.strip()}")


In [ ]:
def model_cache_present(cache_dir, hub, name, min_bytes=100 * 1024 * 1024):
    cache_dir = Path(cache_dir)
    base = cache_dir / "hub" / ("models--" + name.replace("/", "--")) if hub == "hf" else cache_dir / "models" / name.replace("/", "--")
    snapshots = [path for path in (base / "snapshots").glob("*") if path.is_dir()] if base.is_dir() else []
    for snapshot in snapshots:
        largest = max((path.stat().st_size for path in snapshot.rglob("*") if path.is_file()), default=0)
        if largest > min_bytes:
            return True
    return False


def model_cache_report(manifest):
    cache = REPO / "models_cache"
    entries = manifest.get("models", {}) if manifest else {}
    result = {}
    for model_id, entry in entries.items():
        hub = entry.get("hub", "ms")
        model_name = entry["model"]
        ready = model_cache_present(cache, hub, model_name)
        vad_name = entry.get("vad_model")
        if ready and vad_name:
            ready = model_cache_present(cache, hub, vad_name, min_bytes=1)
        result[model_id] = ready
    return result


if MODE == "deploy":
    _cache_state = model_cache_report(SELECTED_MANIFEST_DATA)
    if all(_cache_state.values()) and not SETTINGS["force_download"]:
        print("Selected model files already exist in the cache; skipping downloads")
    else:
        _download_env = RUNTIME_ENV.copy()
        _download_env["MODELSCOPE_CACHE"] = str(REPO / "models_cache")
        _download_env["HF_HOME"] = str(REPO / "models_cache")
        _download_args = [
            str(PYTHON), str(REPO / "scripts" / "download-models.py"),
            "--cache-dir", str(REPO / "models_cache"),
        ]
        for _model_id in SELECTED_MANIFEST_DATA["models"]:
            _download_args.extend(["--model", _model_id])
        print(f"Downloading selected model(s): {', '.join(SELECTED_MANIFEST_DATA['models'])} ...")
        subprocess.run(_download_args, env=_download_env, check=True)


In [ ]:
def install_cloudflared():
    if platform.machine().lower() not in {"x86_64", "amd64"}:
        raise RuntimeError(f"cloudflared download target is Linux x86_64; got {platform.machine()}")
    if not CLOUDFLARED.is_file():
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        temporary = CLOUDFLARED.with_suffix(".download")
        print("Downloading Cloudflare's official cloudflared connector ...")
        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as output:
            shutil.copyfileobj(response, output)
        os.chmod(temporary, 0o755)
        temporary.replace(CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
    version = subprocess.check_output([str(CLOUDFLARED), "--version"], text=True, stderr=subprocess.STDOUT).strip()
    print(version)


def write_secret_files():
    secret_values = {}
    if "ferrum" in PROTOCOLS:
        secret_values.update({
            "CSG_AUTH_SECRET": SETTINGS["auth_secret"],
            "CSG_ENCRYPTION_KEY": SETTINGS["encryption_key"],
        })
    if "deepl" in PROTOCOLS:
        secret_values.update({
            "CSG_TRANSLATE_UPSTREAM_KEY": SETTINGS["translate_upstream_key"],
            "CSG_TRANSLATE_API_KEY": SETTINGS["translate_api_key"],
        })
    if "libretranslate" in PROTOCOLS:
        secret_values.update({
            "CSG_LIBRETRANSLATE_UPSTREAM_KEY": SETTINGS["libretranslate_upstream_key"],
            "CSG_LIBRETRANSLATE_API_KEY": SETTINGS["libretranslate_api_key"],
        })
    body = "# Generated from colab-sg.ipynb; file mode is 0600.\n"
    body += "".join(f"{key}={shlex.quote(str(value))}\n" for key, value in secret_values.items() if value)
    write_private_text(SECRETS_ENV, body)

    if SETTINGS["tunnel"] == "token":
        token = str(SETTINGS["tunnel_token"]).strip()
        if not token:
            raise ValueError("tunnel='token' requires CONFIG['tunnel_token']")
        write_private_text(TUNNEL_TOKEN_FILE, token + "\n")
    elif TUNNEL_TOKEN_FILE.exists():
        TUNNEL_TOKEN_FILE.unlink()
    print("Secret files refreshed with mode 0600 (values not displayed)")


if MODE == "deploy":
    if SETTINGS["tunnel"] != "off":
        install_cloudflared()
    write_secret_files()


In [ ]:
DIRECT_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def _read_pid(path):
    try:
        return int(Path(path).read_text(encoding="ascii").strip())
    except (OSError, ValueError):
        return None


def _pid_alive(pid):
    if not pid:
        return False
    try:
        os.kill(pid, 0)
        return True
    except ProcessLookupError:
        return False
    except PermissionError:
        return True


def _process_command(pid):
    try:
        return Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\\0", b" ").decode(errors="replace")
    except OSError:
        return ""


def _matches_process(pid, marker):
    if not _pid_alive(pid):
        return False
    return marker in _process_command(pid)


def _stop_process(pid_path, marker):
    pid_path = Path(pid_path)
    pid = _read_pid(pid_path)
    if not pid:
        pid_path.unlink(missing_ok=True)
        return
    if not _matches_process(pid, marker):
        print(f"Ignoring stale {pid_path.name} (pid {pid} does not match expected process)")
        pid_path.unlink(missing_ok=True)
        return
    try:
        os.killpg(pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    except OSError:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    deadline = time.monotonic() + 10
    while time.monotonic() < deadline and _pid_alive(pid):
        time.sleep(0.25)
    if _pid_alive(pid):
        try:
            os.killpg(pid, signal.SIGKILL)
        except OSError:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    pid_path.unlink(missing_ok=True)


def _get_health(url, timeout=5):
    try:
        with DIRECT_OPENER.open(url, timeout=timeout) as response:
            if response.status != 200:
                return None
            payload = response.read(64 * 1024)
        return json.loads(payload.decode("utf-8"))
    except (OSError, ValueError, urllib.error.URLError, urllib.error.HTTPError):
        return None


def gateway_health():
    return _get_health(f"http://127.0.0.1:{PORT}/health", timeout=4)


def _tunnel_url():
    try:
        return RUN.joinpath("tunnel.url").read_text(encoding="utf-8").strip()
    except OSError:
        return ""


def tunnel_health():
    url = _tunnel_url()
    return _get_health(url.rstrip("/") + "/health", timeout=8) if url else None


def _wait_for(check, seconds, label):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        result = check()
        if result:
            return result
        time.sleep(2)
    raise TimeoutError(f"{label} did not become healthy within {seconds}s; see logs under {RUN}")


def _gateway_args():
    args = [
        str(PYTHON), "-m", "gateway",
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--device", str(SETTINGS["device"]),
        "--cache-dir", str(REPO / "models_cache"),
        "--preload", *list(SELECTED_MANIFEST_DATA.get("preload", [])),
    ]
    if SETTINGS["max_loaded_models"] is not None:
        args.extend(["--max-loaded-models", str(SETTINGS["max_loaded_models"])])
    if "ferrum" in PROTOCOLS:
        if SETTINGS["auth_secret"]:
            args.extend(["--auth-secret", SETTINGS["auth_secret"]])
        if SETTINGS["encryption_key"]:
            args.extend(["--encryption-key", SETTINGS["encryption_key"]])
    if "deepl" in PROTOCOLS:
        if SETTINGS["translate_upstream"]:
            args.extend(["--translate-upstream", SETTINGS["translate_upstream"]])
            if SETTINGS["translate_upstream_key"]:
                args.extend(["--translate-upstream-key", SETTINGS["translate_upstream_key"]])
        if SETTINGS["translate_api_key"]:
            args.extend(["--translate-api-key", SETTINGS["translate_api_key"]])
    if "libretranslate" in PROTOCOLS:
        if SETTINGS["libretranslate_upstream"]:
            args.extend(["--libretranslate-upstream", SETTINGS["libretranslate_upstream"]])
            if SETTINGS["libretranslate_upstream_key"]:
                args.extend(["--libretranslate-upstream-key", SETTINGS["libretranslate_upstream_key"]])
        if SETTINGS["libretranslate_api_key"]:
            args.extend(["--libretranslate-api-key", SETTINGS["libretranslate_api_key"]])
    if PROTOCOLS & {"deepl", "libretranslate"}:
        args.extend(["--translate-free", str(SETTINGS["translate_free"])])
    else:
        args.extend(["--translate-free", "none"])
    return args


def _digest(value):
    serialized = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def _gateway_fingerprint():
    manifest_bytes = SELECTED_MANIFEST.read_bytes()
    manifest_hash = hashlib.sha256(manifest_bytes).hexdigest()
    return _digest({"args": _gateway_args(), "manifest": manifest_hash})


def _tunnel_fingerprint():
    token_hash = hashlib.sha256(str(SETTINGS["tunnel_token"]).encode("utf-8")).hexdigest()
    return _digest({
        "mode": SETTINGS["tunnel"],
        "name": SETTINGS["tunnel_name"],
        "hostname": SETTINGS["tunnel_hostname"],
        "port": PORT,
        "token_hash": token_hash,
    })


def start_gateway():
    fingerprint = _gateway_fingerprint()
    current_health = gateway_health()
    current_fingerprint = (RUN / "gateway.config.sha256").read_text().strip() if (RUN / "gateway.config.sha256").is_file() else ""
    if current_health and current_fingerprint == fingerprint:
        print("gateway already healthy with current configuration")
        return
    if current_health:
        pid = _read_pid(RUN / "gateway.pid")
        if not pid or not _matches_process(pid, "gateway"):
            raise RuntimeError("A healthy gateway is running but is not owned by this notebook; stop it before redeploying")
        print("Gateway configuration changed; restarting it ...")
    _stop_process(RUN / "gateway.pid", "gateway")
    RUN.mkdir(parents=True, exist_ok=True)
    env = RUNTIME_ENV.copy()
    env["MODELSCOPE_CACHE"] = str(REPO / "models_cache")
    env["HF_HOME"] = str(REPO / "models_cache")
    with (RUN / "gateway.log").open("wb") as log_file:
        process = subprocess.Popen(
            _gateway_args(),
            cwd=REPO,
            env=env,
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    (RUN / "gateway.pid").write_text(f"{process.pid}\n", encoding="ascii")
    write_private_text(RUN / "gateway.config.sha256", fingerprint + "\n")
    print(f"gateway started (pid {process.pid}); waiting for /health ...")
    _wait_for(gateway_health, 900, "gateway")


def stop_gateway():
    _stop_process(RUN / "gateway.pid", "gateway")
    print("gateway stopped")


def start_tunnel():
    mode = str(SETTINGS["tunnel"])
    if mode == "off":
        stop_tunnel()
        print("tunnel disabled")
        return
    fingerprint = _tunnel_fingerprint()
    current_health = tunnel_health()
    current_fingerprint = (RUN / "tunnel.config.sha256").read_text().strip() if (RUN / "tunnel.config.sha256").is_file() else ""
    if current_health and current_fingerprint == fingerprint:
        print(f"tunnel already healthy with current configuration: {_tunnel_url()}")
        return
    if current_health:
        pid = _read_pid(RUN / "cloudflared.pid")
        if not pid or not _matches_process(pid, "cloudflared"):
            raise RuntimeError("A healthy Cloudflare tunnel is running but is not owned by this notebook; stop it before redeploying")
        print("Tunnel configuration changed; restarting it ...")
    if not gateway_health():
        raise RuntimeError("Start the gateway before starting the Cloudflare tunnel")

    _stop_process(RUN / "cloudflared.pid", "cloudflared")
    RUN.mkdir(parents=True, exist_ok=True)
    (RUN / "tunnel.url").unlink(missing_ok=True)
    origin = f"http://127.0.0.1:{PORT}"
    if mode == "token":
        hostname = str(SETTINGS["tunnel_hostname"]).strip()
        if not hostname or "://" in hostname:
            raise ValueError("token mode requires tunnel_hostname without an https:// prefix")
        if not TUNNEL_TOKEN_FILE.is_file():
            raise ValueError("tunnel token file is missing; configure tunnel_token in CONFIG")
        args = [
            str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--loglevel", "info", "run",
            "--token-file", str(TUNNEL_TOKEN_FILE), "--url", origin,
        ]
        known_url = f"https://{hostname}"
    elif mode == "named":
        hostname = str(SETTINGS["tunnel_hostname"]).strip()
        name = str(SETTINGS["tunnel_name"]).strip()
        if not name or not hostname or "://" in hostname:
            raise ValueError("named mode requires tunnel_name and tunnel_hostname (hostname without https://)")
        if not (Path.home() / ".cloudflared" / "cert.pem").is_file():
            raise FileNotFoundError("named mode requires ~/.cloudflared/cert.pem in the Colab VM")
        args = [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "run", "--url", origin, name]
        known_url = f"https://{hostname}"
    elif mode == "quick":
        args = [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--url", origin]
        known_url = ""
    else:
        raise ValueError(f"Unsupported tunnel mode: {mode}")

    with (RUN / "tunnel.log").open("wb") as log_file:
        process = subprocess.Popen(
            args,
            cwd=ROOT,
            env=RUNTIME_ENV,
            stdin=subprocess.DEVNULL,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    (RUN / "cloudflared.pid").write_text(f"{process.pid}\n", encoding="ascii")
    print(f"cloudflared started (pid {process.pid}, mode={mode}); waiting for public URL ...")

    if known_url:
        write_private_text(RUN / "tunnel.url", known_url + "\n")
    else:
        deadline = time.monotonic() + 120
        pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
        while time.monotonic() < deadline:
            log_text = (RUN / "tunnel.log").read_text(encoding="utf-8", errors="replace")[-200_000:]
            match = pattern.search(log_text)
            if match:
                write_private_text(RUN / "tunnel.url", match.group(0) + "\n")
                break
            if not _pid_alive(process.pid):
                raise RuntimeError(f"cloudflared exited; see {RUN / 'tunnel.log'}")
            time.sleep(2)
        else:
            raise TimeoutError(f"Quick Tunnel URL not found; see {RUN / 'tunnel.log'}")
    write_private_text(RUN / "tunnel.config.sha256", fingerprint + "\n")
    _wait_for(tunnel_health, 180, "Cloudflare tunnel")


def stop_tunnel():
    _stop_process(RUN / "cloudflared.pid", "cloudflared")
    (RUN / "tunnel.url").unlink(missing_ok=True)
    print("tunnel stopped")


def show_status():
    gateway = gateway_health()
    url = _tunnel_url()
    tunnel = tunnel_health() if url else None
    gateway_pid = _read_pid(RUN / "gateway.pid")
    tunnel_pid = _read_pid(RUN / "cloudflared.pid")
    print(f"gateway: {'UP' if gateway else 'DOWN'} (pid {gateway_pid or '-'})")
    print(f"tunnel:  {'UP' if tunnel else 'DOWN'} (mode={SETTINGS['tunnel']}, pid {tunnel_pid or '-'})")
    print(f"url:     {url or '(none)'}")
    print(f"health:  {json.dumps(gateway, ensure_ascii=False) if gateway else '(unavailable)'}")
    print(f"logs:    {RUN / 'gateway.log'} | {RUN / 'tunnel.log'}")
    manifest = None
    for manifest_path in (SELECTED_MANIFEST, REPO / "models.json", UPSTREAM_MANIFEST):
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            break
        except (OSError, ValueError):
            continue
    if manifest:
        state = model_cache_report(manifest)
        print("models:")
        for model_id, ready in state.items():
            print(f"  {model_id}: {'OK' if ready else 'MISSING'}")
    return gateway, tunnel


In [ ]:
def _demo_request(label, url, data=None, headers=None, timeout=60):
    request = urllib.request.Request(url, data=data, headers=headers or {}, method="POST")
    try:
        with DIRECT_OPENER.open(request, timeout=timeout) as response:
            body = response.read(1500).decode("utf-8", errors="replace")
            print(f"{label}: HTTP {response.status} | {body}")
            return response.status
    except urllib.error.HTTPError as error:
        body = error.read(1500).decode("utf-8", errors="replace")
        print(f"{label}: HTTP {error.code} | {body}")
        return error.code
    except Exception as error:
        print(f"{label}: ERROR {type(error).__name__}: {error}")
        return None


def _multipart_body(fields, file_bytes, filename):
    boundary = "----colabsg" + os.urandom(12).hex()
    chunks = []
    for name, value in fields:
        chunks.append(
            f"--{boundary}\r\nContent-Disposition: form-data; name=\"{name}\"\r\n\r\n{value}\r\n".encode()
        )
    chunks.append(
        f"--{boundary}\r\nContent-Disposition: form-data; name=\"file\"; filename=\"{filename}\"\r\nContent-Type: audio/wav\r\n\r\n".encode()
    )
    chunks.extend([file_bytes, f"\r\n--{boundary}--\r\n".encode()])
    return boundary, b"".join(chunks)


def run_demo():
    sample = REPO / "bench" / "audio" / "ja" / "short.wav"
    if not sample.is_file():
        url = "https://raw.githubusercontent.com/canxin121/subtitle-gateway/main/bench/audio/ja/short.wav"
        print("Downloading the upstream Japanese demo sample ...")
        with urllib.request.urlopen(url, timeout=60) as response:
            sample.write_bytes(response.read())
    audio = sample.read_bytes()
    base = f"http://127.0.0.1:{PORT}"
    model = next(iter(SELECTED_MANIFEST_DATA["models"]))
    print("Demo sample:", len(audio), "bytes; model:", model)
    print("health:", json.dumps(gateway_health(), ensure_ascii=False))

    if "openai" in PROTOCOLS:
        fields = [("model", model)]
        boundary, body = _multipart_body(fields, audio, sample.name)
        _demo_request(
            "OpenAI transcription",
            base + "/v1/audio/transcriptions",
            body,
            {"Content-Type": f"multipart/form-data; boundary={boundary}"},
            timeout=300,
        )
        fields = [("model", model), ("response_format", "verbose_json"), ("timestamp_granularities[]", "segment")]
        boundary, body = _multipart_body(fields, audio, sample.name)
        _demo_request(
            "OpenAI verbose_json segments",
            base + "/v1/audio/transcriptions",
            body,
            {"Content-Type": f"multipart/form-data; boundary={boundary}"},
            timeout=300,
        )

    if "ferrum" in PROTOCOLS:
        headers = {"x-model": model, "x-compression": "wav"}
        if SETTINGS["auth_secret"]:
            headers["x-auth-token"] = hashlib.sha256(SETTINGS["auth_secret"].encode()).hexdigest()
        _demo_request("ferrum transcription", base + "/transcribe", audio, headers, timeout=300)

    if "deepl" in PROTOCOLS:
        headers = {"Content-Type": "application/json"}
        if SETTINGS["translate_api_key"]:
            headers["Authorization"] = "DeepL-Auth-Key " + SETTINGS["translate_api_key"]
        payload = {"text": ["Good morning"], "target_lang": "ZH"}
        _demo_request("DeepL-compatible translation", base + "/v1/translate", json.dumps(payload).encode(), headers)

    if "libretranslate" in PROTOCOLS:
        payload = {"q": "Hello world", "source": "en", "target": "zh"}
        if SETTINGS["libretranslate_api_key"]:
            payload["api_key"] = SETTINGS["libretranslate_api_key"]
        _demo_request(
            "LibreTranslate translation",
            base + "/translate",
            json.dumps(payload).encode(),
            {"Content-Type": "application/json"},
        )


In [ ]:
if MODE == "deploy":
    print("Starting gateway ...")
    start_gateway()
    print("Starting Cloudflare tunnel ...")
    start_tunnel()
    show_status()
    if SETTINGS["run_demo"]:
        run_demo()
elif MODE == "status":
    show_status()
elif MODE == "stop":
    stop_tunnel()
    stop_gateway()
    show_status()
elif MODE == "uninstall":
    if not SETTINGS["confirm_uninstall"]:
        print(f"Nothing removed. Set CONFIG['confirm_uninstall'] = True to delete {ROOT}.")
    else:
        stop_tunnel()
        stop_gateway()
        if ROOT.exists():
            shutil.rmtree(ROOT)
        print(f"Removed VM deployment directory: {ROOT}")


## 服务模式与日常操作

- 查看状态：把 `CONFIG["mode"]` 改为 `"status"`，再运行上面的 `colab exec` 命令。
- 停止 VM 内 gateway/tunnel：设为 `"stop"`。Colab GPU 会话仍保持运行。
- 清除 `/content/csg` 下的 checkout、venv、模型缓存与密钥文件：先停止本机 launchd/systemd 服务，再设为 `"uninstall"` 且 `confirm_uninstall=True`。这不会释放 Colab 会话。
- 再部署：恢复 `mode="deploy"`；重复执行会复用依赖和缓存。依赖或模型需要强制重装/下载时分别设 `force_deps=True`、`force_download=True`。

模板在 `templates/`。只需在本机安装该服务后，服务会每 120 秒确认/创建会话并重跑 notebook；因此 Colab VM 被回收后也会重新部署。`colab exec` 执行时间上限要足够长（首次装依赖与下模型建议 `--timeout 21600`）。

模型范围、协议开关和翻译上游均由上方 `CONFIG` 控制。模型文件只下载所选项；未选协议的凭据不会传给 gateway。所有 API 路由仍由上游应用挂载，未启用的翻译协议通过 `--translate-free none` 等参数关闭其后端能力。

> `status` 在 VM 上显示 gateway、tunnel 与模型缓存；Colab 会话本身的状态由本机服务模板里的 `colab status -s <session>` 检查。